#Import Libraries & Load Data

In [2]:
import pandas as pd
import numpy as np

# Load the messy dataset
file_path = "Day11_Messy_Company_Employee_Dataset.csv"
df = pd.read_csv(file_path)

# Display the first few rows
df.head()

,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,NaN,Data Scientist,48.0,Other,108371.0,13.2,2017-06-13,Pune,5.0,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31.0,Female,108824.0,12.1,2021-09-15,Mumbai,2.0,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28.0,Female,119400.0,13.2,2021-02-05,Bengaluru,3.0,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30.0,Female,111847.0,NaN,2018-08-05,Jaipur,4.0,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25.0,Female,78677.0,10.9,2018-07-24,Hyderabad,4.0,Hybrid


#Inspecting the Dataset

In [3]:
print("--- Initial Data Information ---")
df.info()

print("\n--- Missing Values Quantification ---")
print(df.isnull().sum())

print("\n--- Duplicate Records Quantification ---")
initial_duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {initial_duplicates}")

print("\n--- Checking for Inconsistent Entries (Categorical Columns) ---")
cat_cols = ['Gender', 'Work_Mode', 'Department', 'City', 'Job_Title']
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

--- Initial Data Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        157 non-null    object 
 1   Employee_Name      157 non-null    object 
 2   Department         152 non-null    object 
 3   Job_Title          157 non-null    object 
 4   Age                153 non-null    float64
 5   Gender             152 non-null    object 
 6   Annual_Salary      152 non-null    float64
 7   Experience_Years   154 non-null    float64
 8   Joining_Date       157 non-null    object 
 9   City               152 non-null    object 
 10  Performance_Score  154 non-null    float64
 11  Work_Mode          155 non-null    object 
dtypes: float64(4), object(8)
memory usage: 14.8+ KB

--- Missing Values Quantification ---
Employee_ID          0
Employee_Name        0
Department           5
Job_Title            0
Age   

#Data Cleaning (Duplicates, Inconsistencies & Data Types)

In [4]:
# 1. Resolve Duplicate Records
df_clean = df.drop_duplicates().copy()
print(f"Dropped {initial_duplicates} duplicate records.")

# 2. Resolve Inconsistent Entries (Capitalization & Whitespace)
for col in cat_cols:
    df_clean[col] = df_clean[col].str.strip().str.title()
print("Fixed inconsistent text formatting (removed trailing spaces, converted to Title Case).")

# 3. Resolve Incorrect Data Types
# Convert Joining_Date from string/object to proper datetime
df_clean['Joining_Date'] = pd.to_datetime(df_clean['Joining_Date'])
print("Converted 'Joining_Date' to datetime object.")

Dropped 7 duplicate records.
Fixed inconsistent text formatting (removed trailing spaces, converted to Title Case).
Converted 'Joining_Date' to datetime object.


#Data Cleaning (Handling Missing Values)

In [5]:
# 1. Numerical Imputation: Using Median
num_cols = ['Age', 'Annual_Salary', 'Experience_Years', 'Performance_Score']
for col in num_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
print("Filled missing numerical values using Median Imputation (robust against outliers).")

# 2. Categorical Imputation: Using Mode
for col in cat_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
print("Filled missing categorical values using Mode Imputation (most frequent class).")

Filled missing numerical values using Median Imputation (robust against outliers).
Filled missing categorical values using Mode Imputation (most frequent class).


#Verification & Export

In [7]:
print("--- Verification: Missing Values After Cleaning ---")
print(df_clean.isnull().sum())

print("\n--- Verification: Duplicate Records After Cleaning ---")
print(f"Total duplicates remaining: {df_clean.duplicated().sum()}")

# Export the final cleaned dataset as a CSV file
export_path = "Cleaned_Company_Employee_Dataset.csv"
df_clean.to_csv(export_path, index=False)

print(f"\nSUCCESS: The dataset condition has improved from having {initial_duplicates} duplicates and multiple missing/inconsistent entries to being perfectly clean. It has been successfully exported as '{export_path}'.")

--- Verification: Missing Values After Cleaning ---
Employee_ID          0
Employee_Name        0
Department           0
Job_Title            0
Age                  0
Gender               0
Annual_Salary        0
Experience_Years     0
Joining_Date         0
City                 0
Performance_Score    0
Work_Mode            0
dtype: int64

--- Verification: Duplicate Records After Cleaning ---
Total duplicates remaining: 0

SUCCESS: The dataset condition has improved from having 7 duplicates and multiple missing/inconsistent entries to being perfectly clean. It has been successfully exported as 'Cleaned_Company_Employee_Dataset.csv'.


# Summary of Data Cleaning Steps

### 1. Initial Assessment
* **Inspection**: Loaded the dataset using `pd.read_csv()` and inspected it using `.info()`, `.isnull().sum()`, and `.duplicated().sum()`.
* **Issues Identified**: The dataset contained 7 duplicate rows, missing values across both numerical and categorical columns, inconsistent text formatting (e.g., mixed cases and extra spaces like 'delhi', 'Delhi ', 'REMOTE', 'remote'), and incorrect data types (dates stored as strings).

### 2. Cleaning Process Applied
* **Handling Duplicates**: Used `drop_duplicates()` to remove the 7 identical rows to prevent skewed analysis.
* **Fixing Inconsistent Entries**: Applied string manipulation techniques (`.str.strip().str.title()`) to categorical columns (`Gender`, `Work_Mode`, `Department`, `City`, `Job_Title`). This removed accidental leading/trailing spaces and unified all text to Title Case.
* **Correcting Data Types**: Converted the `Joining_Date` column from a string/object data type to a proper datetime object using `pd.to_datetime()`.
* **Handling Missing Values**:
  * **Numerical Data (`Age`, `Annual_Salary`, `Experience_Years`, `Performance_Score`)**: Used `fillna()` with the **median** value of each column. *Decision Rationale*: Median imputation was chosen over mean because it is far more robust against outliers (like exceptionally high salaries or ages) that could otherwise skew the data.
  * **Categorical Data (`Gender`, `Work_Mode`, `Department`, `City`)**: Used `fillna()` with the **mode** (the most frequent value). *Decision Rationale*: Mode imputation is the standard and most logical approach for filling missing nominal data where calculating an average is impossible.

### 3. Final Verification and Export
* **Verification**: Ran a final check confirming that there are now 0 missing values and 0 duplicate rows.
* **Export**: The pristine dataset was successfully exported as `Cleaned_Company_Employee_Dataset.csv`, completely ready for exploratory data analysis (EDA) or machine learning modeling.